# Credit Card Customer Churn Prediction
### Using Logistic Regression

**Objective:** Predict whether a credit card customer will churn based on financial and behavioural features.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import pickle
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline


## 2. Load Dataset

In [ ]:
df = pd.read_csv("churn_data.csv")
df.head()


## 3. Exploratory Data Analysis (EDA)

### 3.1 Shape and Data Types

In [ ]:
df.shape

In [ ]:
df.dtypes

### 3.2 Summary Statistics

In [ ]:
df.describe()

### 3.3 Missing Values

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())


### 3.4 Churn Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn', palette='Set2')
plt.title("Churn Count (0 = No Churn, 1 = Churn)")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.show()


### 3.5 Histograms of All Numeric Features

In [ ]:
df.drop(['CustomerID', 'Churn'], axis=1).hist(bins=25, figsize=(12, 8), layout=(2, 4))
plt.suptitle("Histogram for each numeric input variable")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


### 3.6 Key Features vs Churn (Boxplots)

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='Churn', y='Avg_Utilization_Ratio', palette='Set2')
plt.title("Avg Utilization Ratio vs Churn")
plt.xlabel("Churn (0=No, 1=Yes)")
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='Churn', y='Late_Payments', palette='Set2')
plt.title("Late Payments vs Churn")
plt.xlabel("Churn (0=No, 1=Yes)")
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='Churn', y='Total_Transactions', palette='Set2')
plt.title("Total Transactions vs Churn")
plt.xlabel("Churn (0=No, 1=Yes)")
plt.show()


### 3.7 Scatter Plot — Utilization vs Late Payments

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='Avg_Utilization_Ratio', y='Late_Payments',
                hue='Churn', palette='Set1', s=60)
plt.title("Avg Utilization Ratio vs Late Payments (coloured by Churn)")
plt.show()


### 3.8 Correlation Heatmap

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(df.drop('CustomerID', axis=1).corr(), annot=True, fmt=".2f",
            cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## 4. Define Features (X) and Target (y)

In [ ]:
X = df.drop(['CustomerID', 'Churn'], axis=1)
y = df['Churn']

X.head()


In [ ]:
y[:5]

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


In [ ]:
print(Counter(y_train), Counter(y_test))

## 6. Feature Scaling (MinMaxScaler)

In [ ]:
scaler = MinMaxScaler()

scaler.fit(X_train)

# Scale the training data
X_train_scaled = scaler.transform(X_train)

# Scale the test data using the same scaler
X_test_scaled = scaler.transform(X_test)


## 7. Train Logistic Regression Model

In [ ]:
lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)

# Intercept and coefficients
print(lr.coef_, lr.intercept_)


## 8. Predict and Evaluate Model Performance

In [ ]:
y_pred = lr.predict(X_test_scaled)

In [ ]:
y_pred

### 8.1 Confusion Matrix

In [ ]:
results = confusion_matrix(y_test, y_pred)
print("Confusion Matrix of test data:")
print(results)


In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(results, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()


### 8.2 Classification Report

In [ ]:
rep = classification_report(y_test, y_pred)
print("Report of test data:")
print(rep)


### 8.3 Accuracy

In [ ]:
print('Accuracy of Logistic regression classifier on training set: {:.2f}'
      .format(lr.score(X_train_scaled, y_train)))
print('Accuracy of Logistic regression classifier on test set: {:.2f}'
      .format(lr.score(X_test_scaled, y_test)))


## 9. Save Model

In [ ]:
with open("churn_model.pkl", "wb") as f:
    pickle.dump({"model": lr, "scaler": scaler}, f)

print("Model saved as churn_model.pkl")


## 10. Verify — Load Model and Predict

In [ ]:
with open("churn_model.pkl", "rb") as f:
    payload = pickle.load(f)

loaded_lr     = payload["model"]
loaded_scaler = payload["scaler"]

# High-risk sample customer
sample = pd.DataFrame([{
    "Age"                   : 35,
    "Annual_Income"         : 30000,
    "Credit_Limit"          : 6000,
    "Total_Transactions"    : 10,
    "Avg_Utilization_Ratio" : 0.88,
    "Late_Payments"         : 7,
    "Tenure_Years"          : 2
}])

sample_scaled = loaded_scaler.transform(sample)
pred = loaded_lr.predict(sample_scaled)[0]
prob = loaded_lr.predict_proba(sample_scaled)[0][1]

print(f"Prediction        : {'CHURN' if pred == 1 else 'NO CHURN'}")
print(f"Churn Probability : {prob:.2%}")
